# Composure

**Composure** measures how well a pitcher maintains command after specific triggering events. The premise: a pitcher who is truly locked in should not let the previous plate appearance — whether it went well or badly — affect how they attack the next hitter. A composed pitcher throws strikes regardless of context.

The score is an aggregation of 14 component statistics, each measuring the rate of an undesirable outcome (walk, ball) following a specific trigger event. All components are normalized within the season and inverted so that **higher = better composure**.

## Notebook structure

| Phase | What runs | Re-run when... | Speed |
|---|---|---|---|
| **0 — Config** | Thresholds, weights, normalization method | Any parameter change | Instant |
| **1 — Data pull** | Fetch pitch-by-pitch from Baseball Savant | New season / first run | Hours |
| **2 — Metric computation** | Compute raw per-pitcher rates from parquet | Changing thresholds or metric definitions | Seconds |
| **3 — Scoring** | Apply weights + normalization, save results | Changing weights or normalization only | Instant |

---
## Metric Definitions

Each metric is a **conditional rate**: given that event X occurred in the previous plate appearance (or the previous pitch), how often does the pitcher produce outcome Y?

---

### After Hard Contact — walks
Hard contact puts a pitcher in an emotionally difficult position: they did their job (got the batter to swing) but the ball found a gap. A composed pitcher resets immediately.

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `walk_after_barrel` | Previous PA: barrel hit (`launch_speed_angle == 6`) | Next PA: walk or HBP | Did giving up the hardest possible contact cause a loss of command? |
| `walk_after_hard_hit` | Previous PA: exit velo ≥ `HARD_HIT_MPH` (default 90 mph) | Next PA: walk or HBP | Broader hard-contact trigger — any well-struck ball |
| `walk_after_high_xba` | Previous PA: xBa ≥ `HIGH_XBA` (default .450) | Next PA: walk or HBP | Quality-of-contact trigger using expected stats rather than raw velo |

### After Hard Contact — immediate balls
A walk takes time to develop — it requires four balls across a full at-bat. "Immediate ball" metrics capture an even faster signal: does the very **first pitch** of the next PA become a ball? This isolates the reflexive reaction to adversity.

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `first_ball_after_barrel` | Previous PA: barrel | First pitch of next PA: ball | Quickest possible composure check — did the pitcher come out attacking? |
| `first_ball_after_hard_hit` | Previous PA: exit velo ≥ `HARD_HIT_MPH` | First pitch of next PA: ball | Same idea, broader hard-contact trigger |
| `first_ball_after_high_xba` | Previous PA: xBa ≥ `HIGH_XBA` | First pitch of next PA: ball | Expected-stats version |

---

### After Unlucky Soft Contact — walks
Soft contact is the flip side: the pitcher got lucky (weak contact, low expected hit value) but failed to capitalize. A mentally weak pitcher may relax after a good outcome — these metrics catch that.

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `walk_after_soft_hit` | Previous PA: exit velo < `SOFT_HIT_MPH` (default 80 mph) | Next PA: walk or HBP | Did getting lucky lead to complacency? |
| `walk_after_low_xba` | Previous PA: xBa < `LOW_XBA` (default .300) | Next PA: walk or HBP | Expected-stats version |

### After Unlucky Soft Contact — immediate balls

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `first_ball_after_soft_hit` | Previous PA: exit velo < `SOFT_HIT_MPH` | First pitch of next PA: ball | Quick reflex check after a lucky out |
| `first_ball_after_low_xba` | Previous PA: xBa < `LOW_XBA` | First pitch of next PA: ball | Expected-stats version |

---

### Additional Metrics

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `hard_hit_after_walk` | Previous PA: walk or HBP | Next PA: exit velo ≥ `HARD_HIT_MPH` | Does issuing a walk leave a pitcher rattled, causing them to groove pitches to the next hitter? |
| `hard_hit_after_soft` | Previous PA: exit velo < `SOFT_HIT_MPH` | Next PA: exit velo ≥ `HARD_HIT_MPH` | Does soft contact create overconfidence, leading to an easy pitch next time? |
| `walk_after_walk` | Previous PA: walk or HBP | Next PA: walk or HBP | Back-to-back walks — the classic on-mound unraveling signal |
| `ball_after_foul` | Current PA: foul on pitch N | Pitch N+1 (same PA): ball | **Within-PA** metric. After nearly escaping an at-bat (foul with two strikes) or extending one, does the pitcher immediately lose the strike zone? |

---

### Classification rules
- **Ball in dirt** (`ball_in_dirt`, `blocked_ball`) → counted as a ball
- **Hit by pitch** (`hit_by_pitch`) → counted as a walk
- **Barrel** = `launch_speed_angle == 6` (Statcast classification; not a tuneable threshold)
- **Foul tip** → excluded from `ball_after_foul` trigger (foul tip with two strikes ends the PA; it is not a near-escape)
- Non-contact outcomes (K, BB, HBP) have `NaN` exit velo → correctly excluded from hard/soft hit triggers

---
## Phase 0 — Config & Setup

All tuneable parameters live here. Change values and re-run this cell, then re-run Phase 2 (if thresholds changed) or Phase 3 (if only weights/normalization changed).

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from tqdm import tqdm
import pybaseball
from pybaseball import statcast_single_game, playerid_reverse_lookup
import warnings
warnings.filterwarnings('ignore')
pybaseball.cache.enable()

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('composure.ipynb')), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

SEASONS = [2023, 2024, 2025]

# ── Metric thresholds (re-run Phase 2 if you change these) ───────────────────
HARD_HIT_MPH  = 90     # exit velo >= this → hard hit
SOFT_HIT_MPH  = 80     # exit velo <  this → soft contact
HIGH_XBA      = 0.450  # xBa >= this → high expected value event
LOW_XBA       = 0.300  # xBa <  this → low expected value event
# Barrel = launch_speed_angle == 6 (Statcast definition; not tuneable)

# ── Pitcher qualification ─────────────────────────────────────────────────────
MIN_PITCHES = 500      # pitchers below this threshold are excluded from scoring

# ── Per-metric weights (re-run Phase 3 if you change these) ──────────────────
# Default 1.0 = equal contribution. Set to 0 to exclude a metric entirely.
# Increase to emphasise a metric in the composite score.
WEIGHTS = {
    # After hard contact
    'walk_after_barrel':          1.0,
    'walk_after_hard_hit':        1.0,
    'walk_after_high_xba':        1.0,
    'first_ball_after_barrel':    1.0,
    'first_ball_after_hard_hit':  1.0,
    'first_ball_after_high_xba':  1.0,
    # After unlucky soft contact
    'walk_after_soft_hit':        1.0,
    'walk_after_low_xba':         1.0,
    'first_ball_after_soft_hit':  1.0,
    'first_ball_after_low_xba':   1.0,
    # Additional
    'hard_hit_after_walk':        1.0,
    'hard_hit_after_soft':        1.0,
    'walk_after_walk':            1.0,
    'ball_after_foul':            1.0,
}

METRIC_COLS = list(WEIGHTS.keys())

# ── Normalization method (re-run Phase 3 if you change this) ─────────────────
# 'zscore'     — (x - mean) / std across pitchers in the season
# 'percentile' — rank-based 0–1 percentile across pitchers
# 'minmax'     — rescale each metric to [0, 1] across pitchers
NORMALIZATION = 'zscore'

print(f'Config loaded. Data dir: {DATA_DIR}')
print(f'Normalization: {NORMALIZATION} | MIN_PITCHES: {MIN_PITCHES}')
print(f'Thresholds — hard hit: >={HARD_HIT_MPH} mph | soft: <{SOFT_HIT_MPH} mph | '
      f'high xBa: >={HIGH_XBA} | low xBa: <{LOW_XBA}')

---
## Phase 1 — Data Pull

Fetches pitch-by-pitch Statcast data for each season via `pybaseball.statcast_single_game()` (one CSV download per game from Baseball Savant) and saves the result to `data/all_pitches_YYYY.parquet`.

**Run once per season.** Subsequent runs detect the parquet file and load from disk instantly. A full season (~2,430 games) takes 1–2 hours on first fetch; the pybaseball disk cache means individual game files are also reused across notebook sessions.

> The 2024 parquet is already cached from a previous run.

In [ ]:
# ── Shared fetch helpers ──────────────────────────────────────────────────────

# Only keep the columns needed for metric computation — drops the rest to save memory.
# Note: pybaseball returns all columns as raw strings (parse_numerics=False internally),
# so numeric columns are cast explicitly via pd.to_numeric(errors='coerce').
KEEP_COLS = [
    'game_pk', 'game_date', 'pitcher', 'player_name',  # player_name = batter
    'at_bat_number', 'pitch_number',
    'pitch_type', 'pitch_name', 'description', 'events',
    'launch_speed',              # exit velocity (mph) — NaN on non-contact
    'launch_speed_angle',        # Statcast contact quality category (6 = barrel)
    'estimated_ba_using_speedangle',  # xBa — NaN on non-contact
    'release_speed',
]
NUMERIC_COLS = [
    'launch_speed', 'launch_speed_angle', 'estimated_ba_using_speedangle',
    'at_bat_number', 'pitch_number', 'game_pk', 'release_speed',
]

def _normalize_raw(df):
    df = df[[c for c in KEEP_COLS if c in df.columns]].copy()
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def _fetch_game(game_pk):
    try:
        df = statcast_single_game(game_pk)
        return _normalize_raw(df) if (df is not None and not df.empty) else None
    except Exception as e:
        print(f'  Error {game_pk}: {e}')
        return None

def _get_game_pks(season):
    """Fetch all completed regular-season gamePks from the MLB Stats API."""
    teams = requests.get('https://statsapi.mlb.com/api/v1/teams?sportId=1').json()['teams']
    pks = set()
    for team in tqdm(teams, desc=f'{season} schedule', leave=False):
        sched = requests.get(
            f'https://statsapi.mlb.com/api/v1/schedule'
            f'?sportId=1&season={season}&teamId={team["id"]}&gameType=R'
        ).json()
        for date in sched.get('dates', []):
            for game in date.get('games', []):
                if game.get('status', {}).get('abstractGameState') == 'Final':
                    pks.add(game['gamePk'])
    return sorted(pks)

def pull_season(season):
    """Load pitch data from parquet cache if available, otherwise fetch and save."""
    path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'{season}: loaded {len(df):,} pitches from cache.')
        return df
    pks = _get_game_pks(season)
    print(f'{season}: fetching {len(pks)} games...')
    results = [_fetch_game(pk) for pk in tqdm(pks, desc=str(season))]
    dfs = [r for r in results if r is not None]
    print(f'  {len(dfs)} ok / {len(pks) - len(dfs)} failed')
    df = pd.concat(dfs, ignore_index=True)
    df.to_parquet(path, index=False)
    print(f'  Saved → {path}')
    return df

print('Fetch helpers defined.')

In [ ]:
# ── 2023 data pull ────────────────────────────────────────────────────────────
pitches_2023 = pull_season(2023)

In [ ]:
# ── 2024 data pull ────────────────────────────────────────────────────────────
pitches_2024 = pull_season(2024)

In [ ]:
# ── 2025 data pull ────────────────────────────────────────────────────────────
pitches_2025 = pull_season(2025)

---
## Phase 2 — Metric Computation

Loads `all_pitches_YYYY.parquet`, applies the thresholds set in Phase 0, and computes the 14 raw per-pitcher rates. Results are saved to `data/pitcher_metrics_YYYY.parquet` — a small file (one row per pitcher) that Phase 3 loads instantly.

**Re-run when:** you change any threshold in Phase 0 (`HARD_HIT_MPH`, `SOFT_HIT_MPH`, `HIGH_XBA`, `LOW_XBA`).

### How the rates are computed

**Cross-PA metrics (1–13):** For each pitcher-game sequence, the previous PA's outcome flags are shifted down by one row (within `game_pk + pitcher` groups, ascending by `at_bat_number`). The first PA a pitcher throws in any game has no prior context and is correctly excluded via `NaN`.

**Within-PA metric (14 — `ball_after_foul`):** The `is_foul` flag is shifted within each `game_pk + pitcher + at_bat_number` group. The first pitch of every PA is excluded.

**Outcome: `NaN` rate** means the pitcher had zero triggered opportunities for that metric in the season (e.g., no barrel was hit against them in a situation where they then started a new PA). These are handled gracefully in Phase 3 via weighted averaging with `skipna=True`.

In [ ]:
# ── Shared metric helpers ─────────────────────────────────────────────────────

def _add_flags(df):
    """
    Add boolean event flags using the thresholds set in Phase 0.
    Non-contact outcomes (K, BB, HBP) have NaN launch_speed — comparisons
    return False, which is the correct behaviour for hard/soft hit flags.
    """
    d = df.copy()
    # Pitch outcome flags
    d['is_ball']     = d['description'].isin(
                           ['ball', 'ball_in_dirt', 'blocked_ball', 'pitchout', 'intent_ball'])
    d['is_walk']     = d['events'].isin(['walk', 'hit_by_pitch'])  # HBP counts as walk
    d['is_foul']     = d['description'] == 'foul'                  # foul_tip excluded
    # Contact quality flags (NaN launch_speed → False)
    d['is_hard_hit'] = d['launch_speed'] >= HARD_HIT_MPH
    d['is_soft_hit'] = d['launch_speed'] <  SOFT_HIT_MPH
    d['is_barrel_flag'] = (d['launch_speed_angle'] == 6            # Statcast barrel definition
                           if 'launch_speed_angle' in d.columns else False)
    d['is_high_xba'] = d['estimated_ba_using_speedangle'] >= HIGH_XBA
    d['is_low_xba']  = d['estimated_ba_using_speedangle'] <  LOW_XBA
    # Filled version used for metrics 11 & 12: non-contact must count as 0, not be excluded
    d['is_hard_hit_filled'] = d['is_hard_hit'].fillna(False)
    return d

def _pa_summary(pitches):
    """
    Extract one row per plate appearance.
    events, launch_speed, xBa, and contact flags are only populated on the
    LAST pitch of each PA in Statcast data — use groupby.last().
    first_pitch_is_ball requires groupby.first().
    """
    p   = pitches.sort_values(['game_pk', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
    grp = ['game_pk', 'pitcher', 'at_bat_number']
    last  = p.groupby(grp).last().reset_index()
    first = (p.groupby(grp)['is_ball'].first()
              .reset_index()
              .rename(columns={'is_ball': 'first_pitch_is_ball'}))
    cols = [
        'game_pk', 'pitcher', 'at_bat_number',
        'is_walk', 'is_hard_hit', 'is_soft_hit', 'is_barrel_flag',
        'is_high_xba', 'is_low_xba', 'is_hard_hit_filled',
    ]
    return last[cols].merge(first, on=grp, how='left')

def _cross_pa_metrics(pa):
    """
    For each pitcher, shift PA-level outcome flags by 1 within each game
    to get the previous PA's context. Then compute conditional rates.
    Groups by (game_pk, pitcher) so cross-game and cross-pitcher rows never contaminate each other.
    """
    pa = pa.sort_values(['game_pk', 'pitcher', 'at_bat_number']).reset_index(drop=True)
    trigger_cols = ['is_walk', 'is_hard_hit', 'is_soft_hit', 'is_barrel_flag',
                    'is_high_xba', 'is_low_xba']
    prev = (
        pa.groupby(['game_pk', 'pitcher'])[trigger_cols]
          .shift(1)  # shift(1) on ascending at_bat_number = previous PA
          .rename(columns={c: f'prev_{c}' for c in trigger_cols})
    )
    pa = pd.concat([pa, prev], axis=1)

    def rate(trigger, outcome):
        """Rate of outcome among PAs where trigger was True in the previous PA."""
        return pa[pa[trigger] == True].groupby('pitcher')[outcome].mean()

    return pd.DataFrame({
        # ── After hard contact ────────────────────────────────────────────
        'walk_after_barrel':          rate('prev_is_barrel_flag', 'is_walk'),
        'walk_after_hard_hit':        rate('prev_is_hard_hit',    'is_walk'),
        'walk_after_high_xba':        rate('prev_is_high_xba',    'is_walk'),
        'first_ball_after_barrel':    rate('prev_is_barrel_flag', 'first_pitch_is_ball'),
        'first_ball_after_hard_hit':  rate('prev_is_hard_hit',    'first_pitch_is_ball'),
        'first_ball_after_high_xba':  rate('prev_is_high_xba',    'first_pitch_is_ball'),
        # ── After unlucky soft contact ────────────────────────────────────
        'walk_after_soft_hit':        rate('prev_is_soft_hit',    'is_walk'),
        'walk_after_low_xba':         rate('prev_is_low_xba',     'is_walk'),
        'first_ball_after_soft_hit':  rate('prev_is_soft_hit',    'first_pitch_is_ball'),
        'first_ball_after_low_xba':   rate('prev_is_low_xba',     'first_pitch_is_ball'),
        # ── Additional ───────────────────────────────────────────────────
        'hard_hit_after_walk':        rate('prev_is_walk',        'is_hard_hit_filled'),
        'hard_hit_after_soft':        rate('prev_is_soft_hit',    'is_hard_hit_filled'),
        'walk_after_walk':            rate('prev_is_walk',        'is_walk'),
    })

def _ball_after_foul(pitches):
    """
    Within-PA metric: among pitches that immediately follow a foul ball
    (within the same PA), what fraction are balls?
    Shift is within (game_pk, pitcher, at_bat_number) to prevent cross-PA leakage.
    """
    p = pitches.sort_values(['game_pk', 'pitcher', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
    p['prev_is_foul'] = p.groupby(['game_pk', 'pitcher', 'at_bat_number'])['is_foul'].shift(1)
    return (
        p[p['prev_is_foul'] == True]
         .groupby('pitcher')['is_ball']
         .mean()
         .rename('ball_after_foul')
    )

def compute_metrics(season):
    """
    Load all_pitches_YYYY.parquet, apply current Phase 0 thresholds,
    compute all 14 raw per-pitcher rates, and save to pitcher_metrics_YYYY.parquet.
    """
    pitches_path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if not os.path.exists(pitches_path):
        raise FileNotFoundError(f'Run the Phase 1 pull cell for {season} first.')

    print(f'{season}: loading pitches...')
    pitches = _add_flags(pd.read_parquet(pitches_path))

    print(f'{season}: computing metrics...')
    pa      = _pa_summary(pitches)
    metrics = _cross_pa_metrics(pa).join(_ball_after_foul(pitches), how='outer')

    # Store pitch count so Phase 3 can apply MIN_PITCHES without reloading raw pitches
    metrics = metrics.join(pitches.groupby('pitcher').size().rename('pitch_count'))

    out = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    metrics.to_parquet(out)
    print(f'{season}: {len(metrics)} pitchers saved → {out}')
    return metrics

print('Metric helpers defined.')

In [ ]:
# ── 2023 metric computation ───────────────────────────────────────────────────
metrics_2023 = compute_metrics(2023)

In [ ]:
# ── 2024 metric computation ───────────────────────────────────────────────────
metrics_2024 = compute_metrics(2024)

In [ ]:
# ── 2025 metric computation ───────────────────────────────────────────────────
metrics_2025 = compute_metrics(2025)

---
## Phase 3 — Scoring

Loads `pitcher_metrics_YYYY.parquet` for all seasons, applies the `WEIGHTS` and `NORMALIZATION` method from Phase 0, and produces the final Composure Score and **Composure+**.

**Re-run just this cell** when tweaking weights or normalization — no need to re-fetch or re-compute metrics.

### Composure Score formula

For each active metric (weight > 0):
1. Normalize the raw rate across all qualified pitchers in the season using the chosen method
2. Invert the result (×−1) so that lower rate = higher normalized value = better composure
3. Multiply by the metric's weight

Final score = sum of weighted normalized values ÷ sum of weights for metrics with data (NaN metrics don't penalise the pitcher — they simply don't contribute to their average).

### Composure+ formula

A plus stat anchored so that **league average = 100**. Computed from the Composure Score via linear rescaling:

```
composure_plus = 100 + ((composure_score − μ) / σ) × 15
```

where μ and σ are the mean and standard deviation of Composure Score across all qualified pitchers in the season. The ×15 scale factor gives a typical range of roughly 55–145, similar to wRC+ or ERA+. A Composure+ of 115 means the pitcher is one standard deviation above league average composure.

In [9]:
def _normalize(series, method):
    """Normalize across pitchers then invert (lower raw rate = higher score)."""
    s = series.copy().astype(float)
    if method == 'zscore':
        normed = (s - s.mean()) / s.std()
    elif method == 'percentile':
        normed = s.rank(pct=True, na_option='keep')  # 0–1
    elif method == 'minmax':
        normed = (s - s.min()) / (s.max() - s.min())
    else:
        raise ValueError(f'Unknown normalization: {method}')
    return normed * -1  # invert: lower rate = better composure


def score_season(season):
    """
    Load pitcher_metrics_YYYY.parquet, apply WEIGHTS + NORMALIZATION,
    compute Composure Score and Composure+, and return a ranked DataFrame.
    Saves composure_scores_YYYY.csv to DATA_DIR.
    """
    path = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Run the Phase 2 compute cell for {season} first.')

    df     = pd.read_parquet(path)
    df     = df[df['pitch_count'] >= MIN_PITCHES].copy()
    active = [m for m in METRIC_COLS if WEIGHTS.get(m, 0) > 0]

    # Normalize and weight each metric
    normed = pd.DataFrame(index=df.index)
    for col in active:
        normed[col] = _normalize(df[col], NORMALIZATION) * WEIGHTS[col]

    # Weighted mean — denominator uses only metrics with data for that pitcher
    weight_sums = normed.notna().multiply([WEIGHTS[c] for c in active]).sum(axis=1)
    df['composure_score'] = normed.sum(axis=1, skipna=True) / weight_sums

    # Composure+ — league average = 100, std of 15 across qualified pitchers
    # Formula: 100 + ((score - μ) / σ) × 15
    score_mean = df['composure_score'].mean()
    score_std  = df['composure_score'].std()
    df['composure_plus'] = (
        100 + (df['composure_score'] - score_mean) / score_std * 15
    ).round(1)

    # Pitcher names via MLBAM ID reverse lookup
    lookup = playerid_reverse_lookup(df.index.tolist(), key_type='mlbam').set_index('key_mlbam')
    df['pitcher_name'] = df.index.map(lookup['name_last'] + ', ' + lookup['name_first'])
    df['season'] = season

    final = (
        df[['season', 'pitcher_name', 'pitch_count', 'composure_plus', 'composure_score'] + active]
        .dropna(subset=['composure_score'])
        .sort_values('composure_plus', ascending=False)
        .reset_index(drop=True)
    )

    out = os.path.join(DATA_DIR, f'composure_scores_{season}.csv')
    final.to_csv(out, index=False)
    print(f'{season}: {len(final)} qualified pitchers → {out}')
    return final


# ── Score all available seasons ───────────────────────────────────────────────
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)

all_scores = {}
for yr in SEASONS:
    if not os.path.exists(os.path.join(DATA_DIR, f'pitcher_metrics_{yr}.parquet')):
        print(f'{yr}: no metrics file yet — run Phase 2 first.')
        continue
    all_scores[yr] = score_season(yr)

for yr, scores in all_scores.items():
    print(f'\n=== {yr} — Top 20 ===')
    display(scores[['pitcher_name', 'pitch_count', 'composure_plus', 'composure_score']].head(20))
    print(f'\n=== {yr} — Bottom 20 ===')
    display(scores[['pitcher_name', 'pitch_count', 'composure_plus', 'composure_score']].tail(20))

2023: 479 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2023.csv
2024: 474 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2024.csv
2025: 479 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2025.csv

=== 2023 — Top 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
0,"martin, chris",736,159.800,1.762
1,"alexander, tyler",679,141.600,1.226
2,"speier, gabe",797,139.300,1.159
3,"borucki, ryan",531,134.400,1.015
4,"ramirez, nick",659,133.700,0.994
5,"stephenson, robert",771,133.000,0.972
6,"kirby, george",2826,131.100,0.917
7,"graterol, brusdar",932,130.100,0.887
8,"scott, tanner",1220,129.900,0.881
9,"anderson, nick",548,129.700,0.877



=== 2023 — Bottom 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
459,"uribe, abner",520,70.200,-0.880
460,"ruiz, josé",825,70.000,-0.885
461,"marte, yunior",700,69.600,-0.896
462,"lamet, dinelson",605,69.100,-0.911
463,"fujinami, shintaro",1411,68.900,-0.916
464,"soriano, josé",704,68.400,-0.931
465,"miller, mason",611,68.200,-0.937
466,"toussaint, touki",1595,68.200,-0.939
467,"saucedo, tayler",791,67.300,-0.964
468,"kuhl, chad",774,65.800,-1.007



=== 2024 — Top 20 ===


...

,pitcher_name,pitch_count,composure_plus,composure_score
454,"muñoz, roddery",1486,73.100,-0.792
455,"diekman, jake",627,71.600,-0.839
456,"rainey, tanner",943,71.600,-0.838
457,"fairbanks, pete",702,70.900,-0.858
458,"boyle, joe",966,70.300,-0.875
459,"nicolas, kyle",954,69.600,-0.897
460,"montero, rafael",683,69.500,-0.901
461,"leasure, jordan",560,69.100,-0.912
462,"soroka, michael",1447,69.100,-0.912
463,"medina, luis",724,69.000,-0.915



=== 2025 — Top 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
0,"dodd, dylan",536,146.300,1.422
1,"rogers, tyler",978,142.400,1.300
2,"martin, chris",651,139.800,1.220
3,"morejón, adrián",1007,134.100,1.046
4,"soto, gregory",1036,133.500,1.027
5,"lee, dylan",1104,131.200,0.958
6,"king, bryan",1046,130.800,0.945
7,"brogdon, connor",853,129.900,0.917
8,"kelly, zack",587,129.300,0.898
9,"kranick, max",529,128.600,0.877



=== 2025 — Bottom 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
459,"waldrep, hurston",887,73.900,-0.801
460,"pressly, ryan",661,72.900,-0.831
461,"boyle, joe",936,72.500,-0.845
462,"henry, cole",969,72.200,-0.852
463,"birdsong, hayden",1195,71.900,-0.863
464,"megill, tylor",1234,71.000,-0.890
465,"zeferjahn, ryan",1039,70.800,-0.896
466,"gonsolin, tony",609,70.600,-0.904
467,"nicolas, kyle",652,67.800,-0.988
468,"booser, cam",572,66.300,-1.035


---
## Multi-year view

Loads saved CSVs to compare composure across seasons. No re-computation needed.

In [10]:
csvs = [
    os.path.join(DATA_DIR, f'composure_scores_{yr}.csv')
    for yr in SEASONS
    if os.path.exists(os.path.join(DATA_DIR, f'composure_scores_{yr}.csv'))
]
all_years = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)

# Pitchers who appear across multiple seasons — track composure over time
print('=== Multi-season pitchers (avg Composure+, min 2 seasons) ===')
multi = (
    all_years.groupby('pitcher_name')
    .agg(
        seasons=('season', 'count'),
        avg_composure_plus=('composure_plus', 'mean'),
        avg_composure_score=('composure_score', 'mean'),
    )
    .query('seasons > 1')
    .sort_values('avg_composure_plus', ascending=False)
)
display(multi.head(30))

=== Multi-season pitchers (avg Composure+, min 2 seasons) ===


,seasons,avg_composure_plus,avg_composure_score
pitcher_name,,,
"martin, chris",3,143.833,1.308
"dodd, dylan",2,133.150,1.005
"rogers, tyler",3,130.800,0.925
"lee, dylan",2,130.450,0.918
"speier, gabe",2,130.000,0.897
"kittredge, andrew",2,129.600,0.889
"morejón, adrián",2,122.200,0.675
"hjelle, sean",2,122.000,0.649
"woodruff, brandon",2,122.000,0.663
